# Lending Club EDA — Data Understanding & Structural Profiling

**Purpose**
- Confirm table row counts at every pipeline stage
- Split all 151 raw columns into numeric vs. categorical (by cast-success rate)
- Profile null% and cardinality for every column

**Where this fits**

| | |
|---|---|
| Position | 1 of 14 EDA notebooks under `notebooks/02_eda/` |
| Data access | Read-only against `data/02_interim/lendingclub.duckdb` |
| Cleaning | None here — happens later, in `notebooks/03_data_cleaning/` |
| Convention | Every code cell has markdown before it (what/why/how) and after it (what the output means, what's next) |

## Cell map

| # | Step |
|---|---|
| 1 | Connect, confirm row counts (raw / matured / windowed) |
| 2 | Numeric vs. categorical split, all 151 raw columns |
| 3 | Eyeball full column lists for miscategorization |
| 4 | Flag ambiguous columns for the cleaning stage |
| 5 | Full null% / cardinality profile (`SUMMARIZE`) |
| 6 | Check whether high-missingness columns carry signal |

**Scope**
- Profiles all 151 raw columns — none picked or dropped yet
- Starting with `02_eda/02_data_quality_integrity.ipynb`, every later notebook narrows to a fixed, named shortlist (see that notebook's cell 1) — that shortlist, not this notebook, drives the rest of the EDA suite

## Cell 1 — Connect & confirm row counts

- Open the interim DuckDB file read-only
- Confirm `raw_mat`, `matured`, `windowed` row counts match what the ingestion notebook produced — a sanity check before profiling anything

**Answers:** do the three staged tables have the row counts the ingestion notebook produced?

In [1]:
import sys, os, duckdb, pandas as pd, numpy as np
sys.path.insert(0, os.path.abspath("../_shared"))
from nb_setup import connect

con, ASSETS_TABLES, ASSETS_PLOTS = connect()  # read-only; creates the asset folders if missing

# row counts at each pipeline stage, plus raw column count
for tbl in ["raw_mat", "matured", "windowed"]:
    n = con.sql(f"SELECT count(*) FROM {tbl}").fetchone()[0]
    print(f"{tbl}: {n:,} rows")
n_cols = len(con.sql("DESCRIBE raw_mat").fetchall())
print(f"raw_mat: {n_cols} columns")

raw_mat: 2,260,701 rows
matured: 1,348,099 rows
windowed: 1,195,879 rows
raw_mat: 151 columns


**Result**

| Table | Rows |
|---|---|
| `raw_mat` | 2,260,701 |
| `matured` | 1,348,099 |
| `windowed` | 1,195,879 |

- 151 raw columns confirmed on `raw_mat` — matches the ingestion notebook's output
- Every downstream notebook works from a subset of these

**Next:** split columns into numeric vs. categorical — nothing arrived typed (everything loaded as `VARCHAR` on purpose, so nothing gets silently mis-cast).

## Cell 2 — Numeric vs. categorical, by cast-success rate

- For every column in `windowed`, check what fraction of non-null values `TRY_CAST`s to `DOUBLE`
- More reliable than trusting column names; flags genuinely ambiguous columns

**Answers:** how many columns are numeric, how many categorical, and are any ambiguous?

In [2]:
cols = [r[0] for r in con.sql("DESCRIBE windowed").fetchall()]
rows = []
# per column: null%, distinct count, and cast-to-DOUBLE success rate
for c in cols:
    r = con.sql(f'''
        SELECT count(*) n, count("{c}") n_notnull,
               count(DISTINCT "{c}") n_distinct,
               count(TRY_CAST("{c}" AS DOUBLE)) n_castable
        FROM windowed
    ''').fetchone()
    n, n_notnull, n_distinct, n_castable = r
    cast_rate = n_castable / n_notnull if n_notnull else 0
    rows.append((c, n_notnull / n, n_distinct, cast_rate))
type_df = pd.DataFrame(rows, columns=["column", "pct_notnull", "n_distinct", "cast_rate"])
# >95% castable -> numeric, <5% -> categorical/text, else flagged mixed (worth a manual look)
type_df["inferred_type"] = np.where(type_df["cast_rate"] > 0.95, "numeric",
                             np.where(type_df["cast_rate"] < 0.05, "categorical/text", "mixed"))
print(type_df["inferred_type"].value_counts())
print()
print("mixed columns (worth a manual look):")
print(type_df[type_df["inferred_type"] == "mixed"].to_string(index=False))

inferred_type
numeric             114
categorical/text     38
Name: count, dtype: int64

mixed columns (worth a manual look):
Empty DataFrame
Columns: [column, pct_notnull, n_distinct, cast_rate, inferred_type]
Index: []


**Result**

| Type | Count |
|---|---|
| Numeric | 114 |
| Categorical/text | 38 |
| Mixed (ambiguous) | 0 |

No columns landed in the "mixed" bucket — confirms the type assumptions the cleaning pipeline and every other EDA notebook rely on.

**Next:** the cast-rate threshold is mechanical — it can still misclassify a column a human would recognize immediately (e.g. a numeric-looking ID or code). Eyeballing the actual lists, then flagging anything that looks wrong, catches what the automated split can't.

## Cell 3 — Eyeball the full column lists

- The 95%/5% cast-rate threshold is mechanical — it can't tell a genuine numeric measure from a numeric-looking identifier or code
- Printing every column in each bucket lets a human check for anything that looks wrong on its face, before moving on

**Answers:** looking at the actual column names, does anything in the numeric or categorical bucket look miscategorized?

In [3]:
# full column lists, one bucket at a time, for manual review
numeric_cols = sorted(type_df.loc[type_df["inferred_type"] == "numeric", "column"])
categorical_cols = sorted(type_df.loc[type_df["inferred_type"] == "categorical/text", "column"])

print(f"numeric ({len(numeric_cols)}):")
print(numeric_cols)
print()
print(f"categorical/text ({len(categorical_cols)}):")
print(categorical_cols)

numeric (114):
['acc_now_delinq', 'acc_open_past_24mths', 'all_util', 'annual_inc', 'annual_inc_joint', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'chargeoff_within_12_mths', 'collection_recovery_fee', 'collections_12_mths_ex_med', 'deferral_term', 'delinq_2yrs', 'delinq_amnt', 'dti', 'dti_joint', 'fico_range_high', 'fico_range_low', 'funded_amnt', 'funded_amnt_inv', 'hardship_amount', 'hardship_dpd', 'hardship_last_payment_amount', 'hardship_length', 'hardship_payoff_balance_amount', 'id', 'il_util', 'inq_fi', 'inq_last_12m', 'inq_last_6mths', 'installment', 'int_rate', 'is_bad', 'last_fico_range_high', 'last_fico_range_low', 'last_pymnt_amnt', 'loan_amnt', 'max_bal_bc', 'mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl', 'mort_acc', 'mths_since_last_delinq', 'mths_since_last_major_derog', 'mths_since_last_record', 'mths_since_rcnt_il', 'mths_since_recent_bc', 'mths_since_recent_bc_dlq', 'mths_since_recent_inq', 'mths_since_recent_revol_delinq', '

**Result**

Two columns in the numeric bucket look miscategorized on inspection, despite passing the cast-rate test:

| Column | Why it's not really numeric |
|---|---|
| `id` | A loan identifier — casts to `DOUBLE` because it's stored as digits, but it's a label, not a magnitude |
| `policy_code` | Only two values (1 or 2) — a category code, not a measured quantity |

Everything else in both lists reads as expected: financial amounts, counts, and rates in numeric; dates, statuses, and free text in categorical.

**Next:** record these two (and anything else found later) somewhere the cleaning notebook can actually check, rather than leaving them as a note in this markdown cell.

## Cell 4 — Flag ambiguous columns for the cleaning stage

- A markdown note here is easy to miss; a file the cleaning notebook actually reads is not
- This cell writes any manually-flagged column + reason to `eda01_ambiguous_overrides.csv`, so `03_data_cleaning/01_cleaning_and_feature_prep.ipynb` can act on it instead of relying on the cast-rate split alone

**Answers:** which columns need to be handled by note, not by cast-rate, and why?

In [4]:
# manual override list -- add (column, note) here for anything cell 3 flagged, or
# anything a later notebook's domain review turns up
AMBIGUOUS_OVERRIDES = [
    ("id", "loan identifier, not a numeric feature -- exclude from modeling despite passing the cast-rate test"),
    ("policy_code", "2-valued category code (1 or 2), not a measured quantity -- treat as categorical"),
]

overrides_df = pd.DataFrame(AMBIGUOUS_OVERRIDES, columns=["column", "note"])
overrides_df.to_csv(os.path.join(ASSETS_TABLES, "eda01_ambiguous_overrides.csv"), index=False)
print(f"ambiguous-column overrides recorded: {len(overrides_df)}")
print(overrides_df.to_string(index=False))

ambiguous-column overrides recorded: 2
     column                                                                                               note
         id loan identifier, not a numeric feature -- exclude from modeling despite passing the cast-rate test
policy_code                   2-valued category code (1 or 2), not a measured quantity -- treat as categorical


**Result**

| column | note |
|---|---|
| `id` | loan identifier, not a numeric feature -- exclude from modeling despite passing the cast-rate test |
| `policy_code` | 2-valued category code (1 or 2), not a measured quantity -- treat as categorical |

Saved to `eda01_ambiguous_overrides.csv` — the cleaning notebook should check this file and treat these two by their note, not by cast-rate alone. This list is meant to grow: any later notebook that spots another miscategorized column should add it here rather than handling it locally.

**Next:** full null%/cardinality profile across every column.

## Cell 5 — Full null% / cardinality profile, every column

- DuckDB's `SUMMARIZE` gives a one-pass null%/min/max/approx-distinct profile per column, in a single query
- Run against `windowed`, covering the complete 151-column raw schema

**Answers:** which columns are the most/least populated, and are any fully null?

In [5]:
# one-pass null%/cardinality profile, every column, full list ordered by null% desc
summ = con.sql("SUMMARIZE windowed").df()
summ["null_pct"] = summ["null_percentage"].astype(float)
summ_sorted = summ[["column_name", "column_type", "null_pct", "approx_unique"]].sort_values("null_pct", ascending=False)
print(summ_sorted.to_string(index=False))
summ_sorted.to_csv(os.path.join(ASSETS_TABLES, "eda01_summ_sorted.csv"), index=False)
print()
print(f"columns with 0% nulls: {(summ_sorted['null_pct']==0).sum()} of {len(summ_sorted)}")
print(f"columns 100% null: {(summ_sorted['null_pct']==100).sum()}")

                               column_name column_type  null_pct  approx_unique
                                 member_id     VARCHAR    100.00              0
                              next_pymnt_d     VARCHAR    100.00              1
orig_projected_additional_accrued_interest     VARCHAR     99.69           3433
       sec_app_mths_since_last_major_derog     VARCHAR     99.65             93
                           hardship_length     VARCHAR     99.52              1
                           hardship_amount     VARCHAR     99.52           4136
                           hardship_status     VARCHAR     99.52              1
                             deferral_term     VARCHAR     99.52              1
                         hardship_end_date     VARCHAR     99.52             27
                           hardship_reason     VARCHAR     99.52              9
                       hardship_start_date     VARCHAR     99.52             27
                             hardship_ty

**Result**

| Metric | Value |
|---|---|
| Fully populated columns | 80 of 152 |
| Fully null columns | 2 (`member_id`, `next_pymnt_d`) |

- `member_id` is scrubbed by Lending Club before publication — dead weight, not signal
- The highest-missingness band (~99.5%+) is entirely hardship/settlement fields — worth understanding on their own terms, not dismissing as "mostly empty"

**Next:** whether that missingness is itself informative, or just structural noise.

## Cell 6 — Does high missingness carry signal?

- A field that's 99.5% null isn't automatically useless — if *whether* it's populated correlates with the outcome, the missingness itself is a feature
- Checking this directly rather than assuming it either way

**Answers:** do loans with a populated hardship/settlement record have a different bad rate than loans without one?

In [6]:
# does having a hardship record at all correlate with the outcome?
hardship_signal = con.sql("""
    SELECT (hardship_type IS NULL) AS hardship_type_is_null,
           count(*) AS n, round(avg(is_bad), 3) AS bad_rate
    FROM windowed GROUP BY 1 ORDER BY 1
""").df()
print("bad rate by whether hardship_type is populated:")
print(hardship_signal.to_string(index=False))
print()

# same check for the debt settlement flag
settlement_signal = con.sql("""
    SELECT debt_settlement_flag, count(*) AS n, round(avg(is_bad), 3) AS bad_rate
    FROM windowed GROUP BY 1 ORDER BY 1
""").df()
print("bad rate by debt_settlement_flag:")
print(settlement_signal.to_string(index=False))
print()

# hardship_flag itself -- check whether it actually varies in this population
hardship_flag_vals = con.sql("SELECT DISTINCT hardship_flag FROM windowed").df()
print(f"distinct hardship_flag values in windowed: {hardship_flag_vals['hardship_flag'].tolist()}")

hardship_signal.to_csv(os.path.join(ASSETS_TABLES, "eda01_hardship_signal.csv"), index=False)
settlement_signal.to_csv(os.path.join(ASSETS_TABLES, "eda01_settlement_signal.csv"), index=False)

bad rate by whether hardship_type is populated:
 hardship_type_is_null       n  bad_rate
                 False    5726     0.705
                  True 1190153     0.203

bad rate by debt_settlement_flag:
debt_settlement_flag       n  bad_rate
                   N 1163542     0.183
                   Y   32337     1.000

distinct hardship_flag values in windowed: ['N']


**Result**

| Group | Populated | n | Bad rate |
|---|---|---|---|
| Hardship record exists (`hardship_type` not null) | Yes | 5,726 | 70.5% |
| No hardship record | No | 1,190,153 | 20.3% |
| Debt settlement flag = Y | Yes | 32,337 | 100.0% |
| Debt settlement flag = N | No | 1,163,542 | 18.3% |

- `hardship_flag` is a dead end for this population — every row in `windowed` shows `N` (it marks a loan *currently* on an active hardship plan; a matured/finished loan is never "currently" anything)
- `hardship_type IS NULL` is the field that actually carries the history

**What these fields are**

| Field group | What it represents | Populated when |
|---|---|---|
| Hardship (`hardship_type`, `hardship_length`, `hardship_amount`, `hardship_start_date`/`hardship_end_date`, ...) | Temporary payment-relief program — reduced or paused payments for a borrower in financial distress | Loan entered a hardship plan |
| Settlement (`settlement_status`, `settlement_amount`, `settlement_percentage`, `settlement_term`, ...) | Negotiated payoff for less than the full balance, via a third-party settlement company | Loan is already severely delinquent |

Both are event-triggered — populated only for the subset of loans that entered that specific process, not the general population.

**Is the missingness itself a signal?**
- Yes, and a strong one
- Hardship record present → 3.5x the base bad rate (70.5% vs. 20.3%)
- Debt settlement flag = Y → 100.0% bad, by construction — settlement is a workout for loans already failing, not a trait observed before the outcome

**Should they be used in modeling?**
- Not as raw predictors — this is a leakage question, not a missingness question
- Both fields only exist because a loan started going bad; they document the outcome unfolding, not a borrower characteristic knowable at origination
- A model trained on `debt_settlement_flag` would just relearn "settled loans are bad" (already true by definition) — and a brand-new loan has no settlement history yet at scoring time anyway
- Same logic applies to `hardship_type` and its associated fields

**What they're good for instead, and when**

| Use | Stage | Notes |
|---|---|---|
| Binary "ever had a hardship/settlement event" indicator (from the null pattern) | Post-hoc, not origination-time | Collections prioritization, loss-given-default modeling on already-troubled loans |
| Exclude from PD feature set, with leakage reasoning documented | `03_data_cleaning/01_cleaning_and_feature_prep.ipynb` | Not just dropped for "too many nulls" |
| Revisit if target needs more granularity than binary `is_bad` | `02_eda/07_target_outcome_objective.ipynb` | Only if that need arises |

**How they'd be treated if ever used**
- The null pattern itself is the feature (`hardship_type IS NULL` as a 0/1 flag) — not the raw amount/date fields
- Those raw fields are only meaningful conditional on the event having happened; imputing them for the 99.5% of loans where nothing happened would fabricate values with no real meaning